<a href="https://colab.research.google.com/github/OluPaul22/gis/blob/main/Main_NJ_RetSup_ExtDem_Rebuild_2022_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NJ retail supply and external demand rebuild, 2022–2024

This notebook sets up the research dataset rebuild. The input is the audited retailer-location workbook, and the target is the 2016–2024 municipality-year panel. The 2022 origins matter because those locations may supply retail access in 2023 and 2024.

The retailer workbook's operation dates are **license-commencement proxies**, except where the evidence identifies an actual sales commencement. Keep the date basis and review flags alongside the results. The existing `optout5`–`optout20` measures describe nearby populations in municipalities that did not opt in. New **no-retail** demand measures must instead use annual retailer counts.

Run the package and import cells first. The Drive mount and input checks are separate so no source file is changed by setup.


## 1. Install packages

In [ ]:
%pip -q install pandas numpy openpyxl pyreadstat geopandas pyogrio shapely pyproj scipy matplotlib statsmodels linearmodels


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.3 MB/s eta 0:00:00


## 2. Import packages

In [ ]:
from pathlib import Path
from datetime import date, datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import openpyxl
import pyreadstat
import pyogrio
import shapely
import pyproj
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

print(f"pandas {pd.__version__} | geopandas {gpd.__version__} | numpy {np.__version__}")
print("Rebuild packages imported successfully.")


pandas 2.2.3 | geopandas 1.2.0 | numpy 2.1.3
Rebuild packages imported successfully.


## 3. Connect Drive when the input files are ready

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 4. Set input and output paths

In [ ]:
PROJECT_DIR = Path('/content/drive/MyDrive/2026_Dr. Hayes_SummerProjects/CannabisGISproject')
GIS_DIR = PROJECT_DIR / 'NJ_Municipal_Boundaries_3424'
GEO_DATA_DIR = GIS_DIR / 'created_datasets'

# Audited workbook is in the existing created_datasets folder.
RETAILER_WORKBOOK = GEO_DATA_DIR / 'Main_NJOperation_Proxies_2022-2024.xlsx'
BASE_PANEL = PROJECT_DIR / 'can_modified_upd.dta'
MUNICIPALITY_REFERENCE = GEO_DATA_DIR / 'mun_geo.csv'
MUNICIPALITY_PAIRS = GEO_DATA_DIR / 'mun_pairs_20mi.csv'

# No output folder is created until the rebuild begins.
REBUILD_DIR = PROJECT_DIR / 'retail_supply_rebuild_2022_2024'

inputs = {
    'audited retailer workbook': RETAILER_WORKBOOK,
    'updated base panel': BASE_PANEL,
    'municipality reference': MUNICIPALITY_REFERENCE,
    'municipality distance matrix': MUNICIPALITY_PAIRS,
}
for label, path in inputs.items():
    print(f"{'FOUND' if path.is_file() else 'MISSING'} | {label} | {path}")


FOUND | audited retailer workbook | /content/drive/MyDrive/2026_Dr. Hayes_SummerProjects/CannabisGISproject/NJ_Municipal_Boundaries_3424/created_datasets/Main_NJOperation_Proxies_2022-2024.xlsx
FOUND | updated base panel | /content/drive/MyDrive/2026_Dr. Hayes_SummerProjects/CannabisGISproject/can_modified_upd.dta
FOUND | municipality reference | /content/drive/MyDrive/2026_Dr. Hayes_SummerProjects/CannabisGISproject/NJ_Municipal_Boundaries_3424/created_datasets/mun_geo.csv
FOUND | municipality distance matrix | /content/drive/MyDrive/2026_Dr. Hayes_SummerProjects/CannabisGISproject/NJ_Municipal_Boundaries_3424/created_datasets/mun_pairs_20mi.csv


## 5. Read-only input inspection

In [ ]:
if all(path.is_file() for path in inputs.values()):
    retailer_sheet = pd.read_excel(RETAILER_WORKBOOK, sheet_name='Main dataset')
    base_panel = pd.read_stata(BASE_PANEL, convert_categoricals=False)
    municipality_ref = pd.read_csv(MUNICIPALITY_REFERENCE, dtype={'muni_id': str})
    municipality_pairs = pd.read_csv(
        MUNICIPALITY_PAIRS,
        dtype={'muni_a_id': str, 'muni_b_id': str},
    )

    print('Retailer rows:', len(retailer_sheet))
    print('Base panel rows:', len(base_panel))
    print('Base panel municipality-years duplicated:',
          base_panel.duplicated(['muni_id', 'year']).sum())
    print('Municipalities:', municipality_ref['muni_id'].nunique())
    print('Distance-matrix pairs:', len(municipality_pairs))
    display(retailer_sheet.head(5))
else:
    print('Place the missing inputs in the paths above before loading data.')


Retailer rows: 196
Base panel rows: 5085
Base panel municipality-years duplicated: 0
Municipalities: 564
Distance-matrix pairs: 47966


,Municipality / locality,Proxy year,Retail address,Retailer,Approval date,Operation-date proxy,Reported open-by date (meeting slide),Renewal decision date,Date basis,Review flag
0,Bellmawr,2022,"640 Creek Rd, Bellmawr, NJ 08031",Curaleaf NJ II,2022-04-11,2022-04-21,NaT,2024-03-13,CRC issuance,Legacy ATC
1,Bloomfield,2022,"26-48 Bloomfield Ave, Bloomfield, NJ 07003",GTI New Jersey LLC,2022-04-11,2022-04-21,NaT,2024-03-13,CRC issuance,Legacy ATC
2,Bordentown,2022,"191 Route 130, Bordentown, NJ 08505",Curaleaf NJ II,2022-10-27,2022-10-31,NaT,2024-09-19,CRC issuance,Legacy ATC
3,Deptford,2022,"1692 Clements Bridge Rd, Deptford, NJ 08096",Columbia Care dba The Cannabist,2022-04-11,2022-04-21,NaT,2024-03-13,CRC issuance,Legacy ATC
4,Eatontown,2022,"59 NJ-35, Eatontown, NJ 07724",Ayr Wellness NJ LLC,2022-05-24,2022-06-06,NaT,2024-04-11,CRC issuance,Legacy ATC


## Rebuild sequence

1. Match each **retail location** and address to a legal New Jersey municipality ID. Audit duplicate licenses, relocations, missing dates, and coverage before assigning zero counts.
2. Expand each location into annual 2022–2024 year-end status, retaining its evidence class. Aggregate to retailer counts by `muni_id` and `year`.
3. Merge counts with the 2016–2024 base panel. Keep the old presentation variables separate. For adult-use retail, verify the pre-2022 zero assumption and do not infer 2022–2024 zeros from an incomplete inventory.
4. Reuse the municipality-centroid distance matrix. For each focal municipality-year and radius, sum populations in **other municipalities with zero retailers**. Call these `no_retail_pop5`, `no_retail_pop10`, `no_retail_pop15`, and `no_retail_pop20`. Preserve `optout5`–`optout20` as distinct policy measures.
5. Create retail-supply × external-demand interactions and competition measures; check counts, joins, distance ordering, and alternative evidence rules before modeling.

The package and input checks above do not execute this rebuild or overwrite prior datasets.


In [ ]:
base_panel = pd.read_stata(BASE_PANEL, convert_categoricals=False)
municipality_ref = pd.read_csv(MUNICIPALITY_REFERENCE, dtype=str)
municipality_pairs = pd.read_csv(MUNICIPALITY_PAIRS, dtype=str)
print('PANEL', base_panel.shape, list(base_panel.columns))
print('PANEL KEY', base_panel[['muni_id','year']].head().to_dict('records'))
print('REFERENCE', municipality_ref.shape, list(municipality_ref.columns), municipality_ref.head(3).to_dict('records'))
print('PAIRS', municipality_pairs.shape, list(municipality_pairs.columns), municipality_pairs.head(3).to_dict('records'))
print('YEARS', base_panel['year'].value_counts().sort_index().to_dict())

PANEL (5085, 15) ['muni_id', 'muni_name', 'year', 'can_rev', 'can_rev_pcap', 'can_active', 'opted_in', 'eff_tax_rate', 'mri_score', 'dem_vote', 'eqprop_val', 'state_rev', 'pop', 'age65_over', 'ihs_can_rev_pcap']
PANEL KEY [{'muni_id': 101, 'year': 2016.0}, {'muni_id': 101, 'year': 2017.0}, {'muni_id': 101, 'year': 2018.0}, {'muni_id': 101, 'year': 2019.0}, {'muni_id': 101, 'year': 2020.0}]
REFERENCE (564, 7) ['muni_id', 'muni_name', 'county', 'muni_label', 'muni_type', 'latitude', 'longitude'] [{'muni_id': '0503', 'muni_name': 'Cape May Point Borough', 'county': 'CAPE MAY', 'muni_label': 'Cape May Point Borough', 'muni_type': 'Borough', 'latitude': '38.93732672689051', 'longitude': '-74.96519625276774'}, {'muni_id': '0512', 'muni_name': 'West Cape May Borough', 'county': 'CAPE MAY', 'muni_label': 'West Cape May Borough', 'muni_type': 'Borough', 'latitude': '38.94226766777276', 'longitude': '-74.93779313642504'}, {'muni_id': '0502', 'muni_name': 'Cape May', 'county': 'CAPE MAY', 'muni_l

In [ ]:
import re
def place_key(value):
    value = re.sub(r'[^a-z0-9 ]', ' ', str(value).lower())
    value = re.sub(r'\b(township|borough|city|town|village|twp)\b', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()
ref = municipality_ref.copy()
ref['key'] = ref['muni_name'].map(place_key)
src = retailer_sheet[['Municipality / locality', 'Retail address']].copy()
src['key'] = src['Municipality / locality'].map(place_key)
counts = ref.groupby('key')['muni_id'].nunique()
print('Unmatched source localities:', sorted(set(src['key']) - set(ref['key'])))
print('Ambiguous matches:', {k:int(counts[k]) for k in set(src['key']) & set(ref['key']) if counts[k] > 1})
print('Panel IDs missing from reference:', sorted(set(base_panel['muni_id'].astype(int).astype(str).str.zfill(4)) - set(ref['muni_id'])))
print('Reference IDs missing from panel:', sorted(set(ref['muni_id']) - set(base_panel['muni_id'].astype(int).astype(str).str.zfill(4))))

Unmatched source localities: ['atco', 'blackwood', 'hewitt', 'kingston', 'marlton', 'mouth ephraim', 'n brunswick', 'n cape may', 'newfoundland', 's bound brook', 'sicklerville', 'somerset', 'turnersville', 'waretown', 'williamstown']
Ambiguous matches: {'lawrence': 2, 'neptune': 2, 'boonton': 2, 'bordentown': 2, 'freehold': 2, 'egg harbor': 2, 'rockaway': 2, 'washington': 6, 'union': 3, 'andover': 2, 'harrison': 2, 'raritan': 2}
Panel IDs missing from reference: ['0429']
Reference IDs missing from panel: []


In [ ]:
def full_key(value):
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', str(value).lower())).strip()

geo = municipality_ref.copy()
geo['muni_id'] = geo['muni_id'].str.zfill(4)
geo['full_key'] = geo['muni_name'].map(full_key)
geo['short_key'] = geo['muni_name'].map(place_key)
full_ids = geo.groupby('full_key')['muni_id'].agg(list).to_dict()
full_ids.pop('bordentown', None)  # Both source sites are in Bordentown Township.
short_ids = geo.groupby('short_key')['muni_id'].agg(list).to_dict()

# Postal localities and same-name municipalities need an address-level crosswalk.
# These are research matches to audit against official municipal records.
postal_ids = {
    'atco':'0435', 'blackwood':'0415', 'hewitt':'1615',
    'kingston':'1808', 'marlton':'0313', 'mouth ephraim':'0425',
    'n brunswick':'1215', 'n cape may':'0505', 'newfoundland':'1615',
    's bound brook':'1819', 'sicklerville':'0415', 'somerset':'1808',
    'turnersville':'0818', 'waretown':'1520', 'williamstown':'0811',
    'andover':'1901', 'boonton':'1401', 'bordentown':'0304',
    'freehold':'1315', 'harrison':'0904', 'lawrence':'1107',
    'neptune':'1334', 'raritan':'1816', 'union':'2019',
    'washington':'2121',
}
retailer_ids = {'Kind Kush':'1434', 'G2 Dispensary':'1434',
                'Holistic Re-Leaf':'1435'}

def map_location(row):
    locality = row['Municipality / locality']
    exact = full_ids.get(full_key(locality), [])
    if len(exact) == 1:
        return pd.Series((exact[0], 'exact legal name', False))
    short = short_ids.get(place_key(locality), [])
    if len(short) == 1:
        return pd.Series((short[0], 'unique normalized name', False))
    if row['Retailer'] in retailer_ids:
        return pd.Series((retailer_ids[row['Retailer']], 'address-specific research crosswalk', True))
    code = postal_ids.get(place_key(locality))
    if code:
        return pd.Series((code, 'postal/locality research crosswalk', True))
    return pd.Series((pd.NA, 'unresolved', True))

locations = retailer_sheet.copy()
locations[['muni_id','mapping_method','address_review_needed']] = locations.apply(map_location, axis=1)
locations = locations.merge(geo[['muni_id','muni_name','county']], on='muni_id', how='left', validate='many_to_one')
print('Locations:',len(locations),'matched:',locations['muni_id'].notna().sum())
print('Research crosswalk rows:',int(locations['address_review_needed'].sum()))
print('Missing legal IDs:',locations.loc[locations['muni_id'].isna(),['Retailer','Retail address']].to_dict('records'))
print(locations.groupby('mapping_method').size().to_dict())
assert locations['muni_id'].notna().all()
assert locations['muni_name'].notna().all()

Locations: 196 matched: 196
Research crosswalk rows: 37
Missing legal IDs: []
{'address-specific research crosswalk': 3, 'exact legal name': 72, 'postal/locality research crosswalk': 34, 'unique normalized name': 87}


In [ ]:
details = pd.read_excel(RETAILER_WORKBOOK, sheet_name='Date details', header=3)
keep = ['Municipality / locality','Retailer','Retail license','Location end evidence',
        'Documented commencement','Renewed expiration','Years subtracted']
locations = locations.merge(details[keep], on=['Municipality / locality','Retailer'],
                            how='left', validate='one_to_one')
locations['proxy_date'] = pd.to_datetime(locations['Operation-date proxy'], errors='coerce')
locations['open_by_date'] = pd.to_datetime(locations['Reported open-by date (meeting slide)'], errors='coerce')
locations['approval_date'] = pd.to_datetime(locations['Approval date'], errors='coerce')
locations['end_date'] = pd.to_datetime(locations['Location end evidence'], errors='coerce')
locations['start_year'] = locations['Proxy year'].astype(int)
locations['year_basis'] = np.where(locations['proxy_date'].notna(),
                                   'license commencement proxy year', 'open-by bound year')
assert (locations.loc[locations['proxy_date'].notna(),'proxy_date'].dt.year ==
        locations.loc[locations['proxy_date'].notna(),'start_year']).all()
assert locations.loc[locations['proxy_date'].isna(),'open_by_date'].notna().all()
assert locations['Retail license'].dropna().duplicated().sum() == 1  # Montclair-to-Wharton relocation

site_year = locations.assign(_join=1).merge(pd.DataFrame({'year':[2022,2023,2024], '_join':1}), on='_join').drop(columns='_join')
site_year['year_end'] = pd.to_datetime(site_year['year'].astype(str) + '-12-31')
site_year['observed_active_at_year_end'] = ((site_year['year'] >= site_year['start_year']) &
    (site_year['end_date'].isna() | (site_year['end_date'] > site_year['year_end'])))
active_sites = site_year[site_year['observed_active_at_year_end']].copy()
retailer_counts = active_sites.groupby(['muni_id','year'],as_index=False).agg(
    retail_supply_observed=('Retailer','size'),
    renewal_estimate_sites=('Date basis',lambda x:(x=='Renewal estimate').sum()),
    crc_issuance_sites=('Date basis',lambda x:(x=='CRC issuance').sum()),
    address_review_sites=('address_review_needed','sum'))

panel = base_panel.copy()
panel['muni_id'] = panel['muni_id'].astype(int).astype(str).str.zfill(4)
panel['year'] = panel['year'].astype(int)
panel = panel.merge(retailer_counts,on=['muni_id','year'],how='left',validate='one_to_one')
panel['retail_supply_observed'] = panel['retail_supply_observed'].fillna(0).astype(int)
panel['retailer_present_observed'] = (panel['retail_supply_observed'] > 0).astype(int)
panel['retail_supply_scope'] = 'lower bound: reviewed CRC locations; completeness not established'
print('Year-end observed sites:',active_sites.groupby('year').size().to_dict())
print('Panel:',panel.shape,'duplicate municipality-years:',panel.duplicated(['muni_id','year']).sum())
print('Bordentown IDs:',locations.loc[locations['Municipality / locality']=='Bordentown','muni_id'].tolist())
print('Relocation:',active_sites.loc[active_sites['Retailer']=='Ascend New Jersey LLC',['year','muni_name']].to_dict('records'))

Year-end observed sites: {2022: 21, 2023: 84, 2024: 195}
Panel: (5085, 21) duplicate municipality-years: 0
Bordentown IDs: ['0304', '0304']
Relocation: [{'year': 2022, 'muni_name': 'Montclair Township'}, {'year': 2023, 'muni_name': 'Montclair Township'}, {'year': 2022, 'muni_name': 'Rochelle Park Township'}, {'year': 2023, 'muni_name': 'Rochelle Park Township'}, {'year': 2024, 'muni_name': 'Rochelle Park Township'}, {'year': 2024, 'muni_name': 'Wharton Borough'}]


In [ ]:
pairs = municipality_pairs[['muni_a_id','muni_b_id','distance_miles']].copy()
pairs['distance_miles'] = pd.to_numeric(pairs['distance_miles'], errors='raise')
pairs = pairs[pairs['muni_a_id'] != pairs['muni_b_id']]
assert not pairs.duplicated(['muni_a_id','muni_b_id']).any()
assert pairs['distance_miles'].ge(0).all()

neighbor_year = panel.loc[panel['year'].between(2022,2024),
    ['muni_id','year','pop','retail_supply_observed']].copy()
neighbor_year['pop'] = pd.to_numeric(neighbor_year['pop'], errors='coerce')
neighbor_year = neighbor_year.rename(columns={'muni_id':'muni_b_id',
    'pop':'neighbor_pop','retail_supply_observed':'neighbor_retailers'})
near = pairs.merge(neighbor_year, on='muni_b_id', how='left', validate='many_to_many')
assert len(near) == len(pairs)*3
assert near['year'].notna().all()
near['year'] = near['year'].astype(int)

for miles in (5,10,15,20):
    subset = near[near['distance_miles'] <= miles].copy()
    subset['included_pop'] = np.where(subset['neighbor_retailers'].eq(0),
                                       subset['neighbor_pop'], 0)
    subset['missing_pop'] = (subset['neighbor_retailers'].eq(0) &
                             subset['neighbor_pop'].isna()).astype(int)
    summaries = subset.groupby(['muni_a_id','year'],as_index=False).agg(
        nearby_pop_without_recorded_retail=('included_pop','sum'),
        missing_neighbor_pop=('missing_pop','sum'),
        nearby_recorded_retailers=('neighbor_retailers','sum'))
    demand_col = f'nearby_pop_without_recorded_retail{miles}'
    competition_col = f'nearby_recorded_retailers{miles}'
    summaries = summaries.rename(columns={'muni_a_id':'muni_id',
        'nearby_pop_without_recorded_retail':demand_col,
        'nearby_recorded_retailers':competition_col})
    summaries.loc[summaries['missing_neighbor_pop'] > 0,demand_col] = np.nan
    summaries = summaries.drop(columns='missing_neighbor_pop')
    panel = panel.merge(summaries,on=['muni_id','year'],how='left',validate='one_to_one')
    valid_geo = panel['muni_id'].isin(geo['muni_id']) & panel['year'].between(2022,2024)
    no_neighbors = valid_geo & panel[competition_col].isna()
    panel.loc[no_neighbors,[demand_col,competition_col]] = 0

print('2022–2024 missing population rows:',int(neighbor_year['neighbor_pop'].isna().sum()))
print('2024 demand missing by radius:',
      {r:int(panel.loc[panel.year==2024,f'nearby_pop_without_recorded_retail{r}'].isna().sum())
       for r in (5,10,15,20)})
print('Panel rows and columns:',panel.shape)
print('Geo reference lacks panel ID 0429; its demand is intentionally missing.')
print(panel.loc[(panel.year==2024)&(panel.retail_supply_observed>0),
      ['muni_name','retail_supply_observed','nearby_pop_without_recorded_retail10']].head(5).to_string(index=False))

2022–2024 missing population rows: 0
2024 demand missing by radius: {5: 1, 10: 1, 15: 1, 20: 1}
Panel rows and columns: (5085, 29)
Geo reference lacks panel ID 0429; its demand is intentionally missing.
      muni_name  retail_supply_observed  nearby_pop_without_recorded_retail10
  Atlantic City                      13                               95775.0
     Buena Boro                       1                               31210.0
Egg Harbor City                       1                               73403.0
 Egg Harbor Twp                       4                               63686.0
 Hammonton Town                       1                               55250.0


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

for miles in (5,10,15,20):
    a = panel.loc[panel.year.between(2022,2024),f'nearby_pop_without_recorded_retail{miles}']
    assert a.dropna().ge(0).all()
    b = panel.loc[panel.year.between(2022,2024),f'nearby_recorded_retailers{miles}']
    assert b.dropna().ge(0).all()
for r1,r2 in ((5,10),(10,15),(15,20)):
    subset = panel[panel.year.between(2022,2024)]
    low = subset[f'nearby_pop_without_recorded_retail{r1}']
    high = subset[f'nearby_pop_without_recorded_retail{r2}']
    valid = low.notna() & high.notna()
    assert (low[valid] <= high[valid]).all()
    low = subset[f'nearby_recorded_retailers{r1}']
    high = subset[f'nearby_recorded_retailers{r2}']
    valid = low.notna() & high.notna()
    assert (low[valid] <= high[valid]).all()
assert active_sites.groupby('year').size().to_dict() == {2022:21,2023:84,2024:195}

screened_panel = panel.copy()
first_year = active_sites.groupby('muni_id')['year'].min().rename('first_retail_year_recorded')
screened_panel = screened_panel.merge(first_year,on='muni_id',how='left',validate='many_to_one')
screened_panel['years_since_first_recorded'] = np.where(
    screened_panel.year >= screened_panel.first_retail_year_recorded,
    screened_panel.year - screened_panel.first_retail_year_recorded, np.nan)
screened_panel['analysis_eligible_geo'] = (screened_panel['muni_id'] != '0429')
screened_panel['retail_x_nrpop10_screen'] = (screened_panel['retail_supply_observed'] *
    screened_panel['nearby_pop_without_recorded_retail10'])

source_links = pd.read_excel(RETAILER_WORKBOOK, sheet_name='Source links', header=3)
urls = source_links.pivot(index=['Municipality / locality','Retailer'],
                          columns='Source type',values='URL').reset_index()
location_review = locations.merge(urls,on=['Municipality / locality','Retailer'],
                                  how='left',validate='one_to_one')
location_review['source_workbook'] = RETAILER_WORKBOOK.name

output_dir = Path('/content/nj_retail_rebuild_preview')
output_dir.mkdir(parents=True,exist_ok=True)
location_review.to_csv(output_dir/'retailer_location_review.csv',index=False)
active_sites.to_csv(output_dir/'recorded_retailer_years.csv',index=False)
screened_panel.to_csv(output_dir/'municipality_panel_screened_2016_2024.csv',index=False)
rename = {'retail_supply_observed':'retail_n_screen',
          'retailer_present_observed':'retail_any_screen',
          'retail_supply_scope':'supply_scope'}
for miles in (5,10,15,20):
    rename[f'nearby_pop_without_recorded_retail{miles}'] = f'nrpop{miles}_screen'
    rename[f'nearby_recorded_retailers{miles}'] = f'nearret{miles}_screen'
statapanel = screened_panel.rename(columns=rename)
statapanel.to_stata(output_dir/'municipality_panel_screened_2016_2024.dta',
                    write_index=False,version=118)
readme = ("PRELIMINARY RESEARCH DATA — NOT THE FINAL PAPER VARIABLES\n"
"The 196-location workbook is not a certified statewide census. Retail counts are recorded-site counts, not verified total active retailers. Nearby population without a recorded retailer is not proof of no retail access. The 37 address-based municipality crosswalk rows need further review. The workbook dates are license-commencement proxies except where documented otherwise. Continued operation is assumed absent end evidence. Demand is computed only for 2022–2024; the extra Pine Valley panel ID 0429 lacks a matching 2024 municipal geometry. The old optout variables measure policy choice and are distinct from no-retail demand. Original sources and date rules are in Main_NJOperation_Proxies_2022-2024.xlsx.\n")
(output_dir/'README.txt').write_text(readme)
with ZipFile(output_dir/'nj_retail_rebuild_preview.zip','w',ZIP_DEFLATED) as z:
    for path in output_dir.iterdir():
        if path.name != 'nj_retail_rebuild_preview.zip': z.write(path,arcname=path.name)
print('Preview files:',[(p.name,p.stat().st_size) for p in output_dir.iterdir()])
print('Mapped retailer rows:',len(location_review),'municipality-year rows:',len(screened_panel))

Preview files: [('README.txt', 775), ('recorded_retailer_years.csv', 94179), ('municipality_panel_screened_2016_2024.csv', 1155709), ('retailer_location_review.csv', 134894), ('nj_retail_rebuild_preview.zip', 477295), ('municipality_panel_screened_2016_2024.dta', 1501410)]
Mapped retailer rows: 196 municipality-year rows: 5085


In [ ]:
from google.colab import files
files.download(str(output_dir / 'nj_retail_rebuild_preview.zip'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation and next audit

This is a **screened research preview**, not a final statewide retailer census. `retail_supply_observed` counts locations found in the audited CRC workbook and continued through each December 31 unless the workbook records an end or relocation. License-commencement dates are proxies for sales opening. The Wharton relocation contributes to 2024 using its documented open-by year, without inventing an exact opening date.

`nearby_pop_without_recorded_retail5/10/15/20` sums the population of **other** municipalities within the specified centroid radius that have no retailer **recorded in this workbook**. It is not yet the memo's verified no-retail external-demand variable. The 37 address-based legal-municipality matches remain flagged in `retailer_location_review.csv`. The existing `optout5`–`optout20` concepts measure municipal policy and should remain separate; new retailer addresses do not alter that policy measure.

The base panel retains ID `0429`, Pine Valley, while the 564-municipality geographic reference does not. [New Jersey Department of Health](https://www.nj.gov/health/hcpnj/county-reports/camden-county.shtml) reports that Pine Valley merged with Pine Hill in January 2022. Its geographic demand values are left missing. The ZIP and component files under `/content/nj_retail_rebuild_preview/` are temporary Colab-runtime outputs; rerun the cells to regenerate them.

Primary source index: [NJ Cannabis Regulatory Commission license awardees and renewals](https://www.nj.gov/cannabis/businesses/license-awardees/). The source workbook provides record-level document links and its date-decision rules.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')